# Image rotation test

Quick visual check of the rotation used by the pipeline. It calls the **real**
`docext_batch.images.tiff_to_jpeg` (the same function the extractor uses), so what
you see here is exactly what the model would receive.

Only needs Pillow + matplotlib (no GPU / docext / vLLM):
```
pip install pillow matplotlib
```
Run this notebook from the repo root (the folder containing `docext_batch/`).

`rotate` is in **degrees clockwise** — the same value you'd put under
`extraction.rotate` in your config.

In [ ]:
import sys
import tempfile
from pathlib import Path

# Make the project's docext_batch package importable.
here = Path.cwd()
root = next((p for p in (here, *here.parents) if (p / 'docext_batch').is_dir()), None)
assert root is not None, 'Run this from inside the repo (needs the docext_batch/ folder).'
sys.path.insert(0, str(root))

from PIL import Image
import matplotlib.pyplot as plt
from docext_batch.images import tiff_to_jpeg, set_max_image_pixels

set_max_image_pixels(300_000_000)  # match the pipeline's decompression-bomb limit
print('project root:', root)

## 1. Pick an image and an angle

Point `IMAGE_PATH` at any TIFF (or PNG/JPG) and set `ANGLE`.

In [ ]:
# --- edit these ---
IMAGE_PATH = root / 'data' / 'input' / 'sample' / 'test.tiff'   # any TIFF/PNG/JPG
ANGLE = -90        # degrees CLOCKWISE (same as extraction.rotate)
MAX_SIZE = 2048   # same as extraction.max_img_size

assert IMAGE_PATH.exists(), f'Not found: {IMAGE_PATH}'
work = Path(tempfile.mkdtemp(prefix='rotation_test_'))
print('using:', IMAGE_PATH)

## 2. Original vs rotated

Both go through the same `tiff_to_jpeg` pipeline (flatten to RGB, rotate, downscale),
so the comparison is fair. Titles show the resulting pixel size.

In [ ]:
orig = tiff_to_jpeg(IMAGE_PATH, work / 'orig', max_size=MAX_SIZE, rotate=0)
rot = tiff_to_jpeg(IMAGE_PATH, work / 'rot', max_size=MAX_SIZE, rotate=ANGLE)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
for ax, path, title in [(axes[0], orig, 'original'), (axes[1], rot, f'rotated {ANGLE} clockwise')]:
    im = Image.open(path)
    ax.imshow(im)
    ax.set_title(f'{title}  -  {im.size[0]}x{im.size[1]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Compare common angles at a glance

0 / 90 / 180 / 270 (right angles are lossless; arbitrary angles get white corners).

In [ ]:
angles = [0, 90, 180, 270]
fig, axes = plt.subplots(1, len(angles), figsize=(5 * len(angles), 6))
for ax, a in zip(axes, angles):
    im = Image.open(tiff_to_jpeg(IMAGE_PATH, work / f'a{a}', max_size=MAX_SIZE, rotate=a))
    ax.imshow(im)
    ax.set_title(f'{a}  -  {im.size[0]}x{im.size[1]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. (Optional) Interactive slider

Needs `ipywidgets` (`pip install ipywidgets`). Drag to find the right angle, then put
that number under `extraction.rotate` in your config.

In [ ]:
try:
    from ipywidgets import interact, IntSlider

    def preview(angle=0):
        im = Image.open(tiff_to_jpeg(IMAGE_PATH, work / f'live{angle}', max_size=MAX_SIZE, rotate=angle))
        plt.figure(figsize=(8, 8))
        plt.imshow(im)
        plt.title(f'rotate {angle} clockwise  -  {im.size[0]}x{im.size[1]}')
        plt.axis('off')
        plt.show()

    interact(preview, angle=IntSlider(min=0, max=359, step=5, value=ANGLE))
except ImportError:
    print('ipywidgets not installed - skip this cell, or: pip install ipywidgets')